# Exploration du découpage en chunks — data/processed/events.json

Avant de choisir `CHUNK_SIZE`/`CHUNK_OVERLAP` dans `scripts/vectorize_events.py`, on regarde la longueur réelle du champ `text` sur les vrais événements — pas une valeur par défaut supposée.

In [ ]:
import json
import statistics
from pathlib import Path

DATA_PATH = Path("..") / "data" / "processed" / "events.json"
events = json.loads(DATA_PATH.read_text(encoding="utf-8"))

lengths = [len(e["text"]) for e in events]
print("Nombre d'événements :", len(lengths))

## 1. Statistiques générales de longueur (en caractères)

In [ ]:
quantiles = statistics.quantiles(lengths, n=100)

print("Min                :", min(lengths))
print("Max                :", max(lengths))
print("Moyenne            :", round(statistics.mean(lengths)))
print("Médiane            :", round(statistics.median(lengths)))
print("25e percentile     :", round(quantiles[24]))
print("75e percentile     :", round(quantiles[74]))
print("90e percentile     :", round(quantiles[89]))
print("95e percentile     :", round(quantiles[94]))
print("99e percentile     :", round(quantiles[98]))

## 2. Répartition par tranche de longueur

In [ ]:
buckets = [(0, 200), (200, 500), (500, 1000), (1000, 2000), (2000, 5000), (5000, None)]

for low, high in buckets:
    if high is None:
        count = sum(1 for length in lengths if length >= low)
        label = f"{low}+"
    else:
        count = sum(1 for length in lengths if low <= length < high)
        label = f"{low}-{high}"
    bar = "#" * (count // 20)
    print(f"{label:<12}{count:>6}  {bar}")

## 3. Combien d'événements dépasseraient CHUNK_SIZE = 1000 caractères (découpage réel en plusieurs chunks) ?

In [ ]:
CHUNK_SIZE = 1000

over_chunk_size = [length for length in lengths if length > CHUNK_SIZE]
print(f"Événements dépassant {CHUNK_SIZE} caractères : {len(over_chunk_size)}/{len(lengths)} "
      f"({len(over_chunk_size)/len(lengths)*100:.1f}%)")